<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum02/BD_KelasA_P02_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BD_KelasA_P02_2411531001_Habibi Putra Rizqullah

**Praktikum 2 - Pengumpulan dan Pra-pemrosesan Data (Data Acquisition & Preprocessing)**


# **K. Langkah Kerja**

## K-1. Import Library dan Inisialisasi

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.2 MB/s eta 0:00:00


Faker tidak terpasang secara bawaan di Google Colab, jadi harus diinstal dulu lewat shell command (`!` di awal baris berarti perintah dijalankan di terminal, bukan sebagai kode Python). Flag `-q` (quiet) dipakai supaya log instalasi tidak memenuhi output sel.

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Empat pustaka yang dipakai sepanjang praktikum:

- `numpy` : operasi numerik dan penyedia nilai `np.nan` untuk menandai data kosong
- `pandas` : pustaka utama untuk mengolah data tabular (DataFrame)
- `Faker` : membangkitkan data palsu yang realistis (nama orang, kota, tanggal)
- `random` : memilih nilai acak dari list, dipakai untuk variasi format harga, tanggal, dan kategori

## K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)

In [3]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


Sel ini mensimulasikan tahap *acquisition*, yaitu proses data masuk dari sumbernya dalam kondisi masih mentah dan belum dibersihkan.

**1. Penguncian seed.** `np.random.seed()`, `random.seed()`, dan `Faker.seed()` dipanggil bertiga karena ketiganya mengendalikan generator angka acak yang berbeda. Kalau salah satu tidak dikunci, hasilnya tidak akan sama persis saat notebook dijalankan ulang.

**2. Perulangan pembuatan 500 baris.** Setiap iterasi membentuk satu transaksi lengkap: ID berformat `TRX00001` (`:05d` artinya angka dipadatkan jadi 5 digit dengan nol di depan), nama pembeli dari Faker lokal Indonesia, nama produk, kategori, harga, dan jumlah barang.

**3. Penyisipan masalah data secara sengaja.** Bagian inilah yang membuat data terasa "kotor" seperti di dunia nyata:
- *Harga* ditulis dalam 4 format berbeda : angka polos (`50000`), pakai prefiks `Rp50.000`, pakai desimal `50000.0`, dan berspasi ` 50000 `
- *Tanggal* ditulis dalam 3 format berbeda : ISO (`2026-07-11`), `11/07/2026`, dan `11-07-2026`
- *Metode bayar* diubah jadi huruf kecil pada sekitar 30% baris
- *Kategori* diubah jadi huruf kapital semua plus spasi berlebih pada sekitar 20% baris
- *Rating* diisi `None` pada sebagian baris, karena rating memang bersifat opsional

**4. Penyuntikan missing value.** `df.sample(frac=..., random_state=SEED)` memilih sebagian kecil baris secara acak namun terkendali, lalu `df.loc[idx, col] = np.nan` mengosongkan kolomnya. Proporsinya 2% untuk `customer_name`, 3% untuk `shipping_city`, dan 1,5% untuk `payment_method`.

**5. Duplikasi.** 15 baris diambil lalu disatukan kembali ke DataFrame dengan `pd.concat()`, meniru transaksi yang tercatat dua kali akibat kegagalan jaringan. Setelah itu `df.sample(frac=1)` mengacak urutan seluruh baris supaya duplikatnya tidak berkumpul di bagian bawah.

**Hasil:** 500 baris dasar + 15 baris duplikat = **515 baris**, disimpan ke `transaksi_mentah.csv`.

In [4]:
df.head()

,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000,4,kartu kredit,2026-07-12,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,Rp500.000,1,E-Wallet,2026-07-08,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000,1,COD,2026-08-24,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,Rp250.000,1,COD,26/07/2026,Tual,4.0
4,TRX00074,NaN,Eius,Olahraga,250000.0,4,NaN,30/06/2026,NaN,1.0


`df.head()` menampilkan 5 baris pertama untuk memeriksa struktur data secara sekilas. Dari sini variasi format yang disengaja tadi sudah mulai terlihat, terutama pada kolom `price` dan `transaction_date` yang isinya belum seragam.

## K-3. Deteksi dan Penanganan Missing Value

In [5]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


 `df.isnull()` mengubah seluruh DataFrame menjadi tabel berisi `True`/`False`, di mana `True` berarti nilainya kosong. `.sum()` kemudian menjumlahkannya per kolom (`True` dihitung sebagai 1), sehingga keluar rekap berapa banyak nilai kosong di tiap kolom.

Hasilnya: `customer_name` 20 kosong, `payment_method` 16 kosong, `shipping_city` 30 kosong, dan `rating` 166 kosong. Kolom `rating` paling banyak kosong karena memang tidak wajib diisi pembeli.

In [6]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")
print("Jumlah baris setelah penanganan missing value:", len(df))

Jumlah baris setelah penanganan missing value: 495


Strategi penanganan **tidak disamaratakan** untuk semua kolom, melainkan ditentukan berdasarkan seberapa penting kolom itu bagi keabsahan sebuah transaksi.

- `dropna(subset=[...])` membuang **barisnya** apabila `customer_name` atau `payment_method` kosong. Dipilih karena tanpa nama pembeli dan metode pembayaran, sebuah transaksi tidak bisa diidentifikasi secara utuh, jadi barisnya memang tidak layak dipakai.
- `fillna("Tidak Diketahui")` mengisi `shipping_city` yang kosong tanpa membuang barisnya. Dipilih karena baris itu masih berguna untuk analisis lain (misal total penjualan per kategori) meski kota pengirimannya tidak diketahui.
- `rating` sengaja **tidak disentuh sama sekali**. Mengisi rating kosong dengan angka tebakan akan mendistorsi perhitungan rating rata-rata nantinya, jadi lebih jujur dibiarkan kosong dan ditangani secara eksplisit saat dianalisis.

**Hasil:** 20 baris terbuang, dari 515 menjadi **495 baris**.

## K-4. Deteksi dan Penanganan Duplicate

In [7]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Ada dua cara mendeteksi duplikat yang dipakai di sini:

- `df.duplicated()` menandai baris yang **seluruh kolomnya** identik dengan baris lain
- `df['transaction_id'].duplicated()` menandai baris yang hanya **ID transaksinya** yang kembar. Pengecekan ini lebih ketat karena ID transaksi seharusnya unik, jadi ID kembar tetap mencurigakan walaupun kolom lainnya berbeda

`drop_duplicates()` kemudian menghapus salinan kedua dan seterusnya, menyisakan satu baris asli.

**Catatan penting pada hasil:** duplikat yang terdeteksi di sini **hanya 5 baris**, padahal pada tahap K-2 yang disisipkan ada 15. Penyebabnya adalah urutan eksekusi: karena `dropna()` di K-3 dijalankan lebih dulu, sebagian pasangan duplikat sudah rusak sebelum sempat dideteksi. Dari 30 baris yang terlibat dalam 15 pasangan duplikat, 20 di antaranya kebetulan punya nilai kosong pada `customer_name` atau `payment_method`, sehingga salah satu atau kedua salinannya sudah terbuang oleh `dropna()`. Inilah alasan praktis mengapa `drop_duplicates()` lebih aman dijalankan di awal pipeline (lihat Pertanyaan Evaluasi no. 2).

**Hasil:** 5 baris terbuang, dari 495 menjadi **490 baris**.

## K-5. Koreksi Tipe Data dan Standardisasi Format

### a. Standardisasi teks kategorikal (category, payment_method, shipping_city)

In [8]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].unique())
print(df["payment_method"].unique())

<StringArray>
['Buku', 'Elektronik', 'Fashion', 'Rumah Tangga', 'Kesehatan', 'Olahraga']
Length: 6, dtype: string
<StringArray>
['Kartu Kredit', 'E-Wallet', 'COD', 'Transfer Bank']
Length: 4, dtype: string


Tiga operasi dirangkai berurutan pada setiap kolom teks:

1. `.astype("string")` mengubah kolom ke tipe string milik pandas. Dipakai **bukan** `.astype(str)` biasa, karena `.astype(str)` akan mengubah nilai `NaN` menjadi teks harfiah `"nan"` sehingga data kosong jadi tidak terdeteksi lagi sebagai kosong
2. `.str.strip()` membuang spasi berlebih di awal dan akhir teks, membersihkan sisa spasi yang disisipkan pada tahap K-2
3. `.str.title()` menyeragamkan kapitalisasi jadi format judul, sehingga `"ELEKTRONIK"`, `"elektronik"`, dan `"Elektronik"` semuanya menjadi satu nilai yang sama

Baris `replace()` di bawahnya adalah koreksi khusus: `.str.title()` mengubah `"COD"` menjadi `"Cod"` karena dianggap kata biasa, padahal COD adalah singkatan (*Cash On Delivery*) yang seharusnya tetap kapital penuh.

**Hasil:** `category` menyisakan 6 nilai unik dan `payment_method` 4 nilai unik, sesuai jumlah kategori dan metode bayar yang didefinisikan di awal.

### b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik)

In [9]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)
df["price"].head()

,price
0,50000.0
1,500000.0
2,25000.0
3,250000.0
5,250000.0


Kolom `price` seharusnya numerik, tapi karena isinya bercampur simbol `Rp`, titik ribuan, dan spasi, pandas membacanya sebagai teks. Fungsi `bersihkan_harga()` membersihkannya per nilai:

- `pd.isna(x)` dicek paling awal agar nilai kosong langsung dikembalikan sebagai `NaN`, tidak ikut diproses jadi teks `"nan"`
- `.strip()` membuang spasi, `.replace("Rp", "")` membuang prefiks mata uang
- `.replace(".", "")` membuang titik ribuan (`50.000` → `50000`), lalu `.replace(",", ".")` mengubah koma desimal gaya Indonesia menjadi titik yang dikenali Python
- `float(x)` dibungkus `try`/`except ValueError` sebagai pengaman: kalau ada nilai yang tetap gagal dikonversi, hasilnya jadi `NaN` dan **tidak membuat seluruh sel error**

`.apply()` menjalankan fungsi ini ke setiap baris kolom `price`. Hasilnya seluruh nilai berhasil dikonversi menjadi angka, tanpa ada yang jatuh jadi `NaN`.

### c. Standardisasi format tanggal ke YYYY-MM-DD

In [10]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
df["transaction_date"].head()

,transaction_date
0,2026-07-12
1,2026-07-08
2,2026-08-24
3,2026-07-26
5,2026-09-14


Kolom tanggal berisi campuran tiga format sekaligus, jadi perlu diseragamkan ke satu format baku `YYYY-MM-DD`.

Cara kerja `parse_tanggal()`: untuk setiap nilai, tiga format dicoba **satu per satu** secara berurutan. Begitu ada format yang cocok, hasilnya langsung dikembalikan; kalau gagal, `except ValueError` menangkapnya dan `continue` melanjutkan ke format berikutnya. Kalau ketiganya gagal, dikembalikan `pd.NaT` (penanda tanggal kosong).

**Mengapa tidak memakai cara yang lebih singkat?** Kode `pd.to_datetime(..., format="mixed", dayfirst=True)` terlihat lebih ringkas, tapi berbahaya di sini. Karena `dayfirst=True` berlaku untuk *semua* nilai, tanggal yang sudah berformat ISO ikut dibalik — misalnya `2026-07-11` yang seharusnya 11 Juli 2026 bisa terbaca jadi 7 November 2026. Yang paling berbahaya, kesalahan ini **tidak memunculkan error sama sekali**, tanggalnya tetap terbaca tapi nilainya salah, sehingga sulit ketahuan kecuali dicek manual.

`.dt.strftime("%Y-%m-%d")` di akhir mengubah hasil datetime kembali menjadi teks berformat seragam.

### d. Finalisasi tipe data

In [11]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)
df.dtypes

,0
transaction_id,object
customer_name,object
product_name,object
category,string[python]
price,float64
quantity,int64
payment_method,string[python]
transaction_date,object
shipping_city,string[python]
rating,float64


Langkah penutup untuk memastikan tipe data setiap kolom sudah benar sebelum diekspor. `quantity` dipaksa menjadi `int` (jumlah barang selalu bilangan bulat) dan `price` menjadi `float64`. `df.dtypes` dipanggil sebagai verifikasi akhir untuk memastikan tidak ada kolom numerik yang masih bertipe `object` (teks).

## K-6. Ekspor Dataset Bersih

In [12]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


 `to_csv()` menyimpan DataFrame yang sudah bersih ke file CSV. Argumen `index=False` penting supaya nomor indeks DataFrame tidak ikut tertulis sebagai kolom tambahan di file hasil.

Nama file `transaksi_bersih.csv` harus dipakai persis seperti ini, karena file ini akan dibaca ulang pada Praktikum 3 dan dikonversi menjadi `transaksi_bersih.parquet`.

**Hasil akhir: 490 baris** — dari 515 baris mentah, total 25 baris terbuang (20 karena data wajib kosong, 5 karena duplikat), atau sekitar 4,9% dari keseluruhan data.

# **O. Analisis Hasil**

Dari 515 baris data mentah, 25 baris (sekitar 4,9%) dibuang selama proses pra-pemrosesan. Rinciannya berbeda dari asumsi awal modul: 20 baris dibuang karena kehilangan data wajib (customer_name atau payment_method), dan 5 baris dibuang karena duplicate murni yang masih terdeteksi setelah data kosong ditangani lebih dulu.

Penyebab pergeseran angka ini karena urutan eksekusi. Duplicate yang sengaja disisipkan pada tahap acquisition sebenarnya ada 15 baris, tapi karena dropna() dijalankan lebih dulu sebelum drop_duplicates(), sebagian pasangan duplicate ikut hilang saat penanganan missing value. Dari 30 baris yang terlibat dalam 15 pasangan duplicate tersebut, 20 di antaranya kebetulan punya nilai kosong pada customer_name atau payment_method, sehingga salah satu atau kedua salinannya sudah terbuang duluan oleh dropna. Saat drop_duplicates() dijalankan setelahnya, yang masih terdeteksi sebagai duplicate murni tinggal 5 baris.

Kehilangan data selama proses cleaning ini normal, bukan tanda ada yang salah. Justru itu bukti proses cleaning bekerja dengan benar, karena baris yang dibuang memang tidak layak dipakai sebagai dasar analisis.

Kolom rating sengaja tidak diisi (imputed). Mengisi rating kosong dengan angka tebakan akan mendistorsi rating rata-rata yang dihitung nanti. Lebih aman membiarkannya kosong dan menanganinya secara eksplisit saat dianalisis, misalnya dengan menghitung rata-rata hanya dari transaksi yang memang punya rating.

# **P. Studi Kasus**

**1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda
lakukan.**

Angka 515 adalah jumlah baris pada data mentah, sebelum melewati proses apa pun, termasuk baris yang datanya tidak lengkap dan baris yang tercatat dua kali akibat kegagalan sistem. Angka 490 adalah jumlah baris setelah data melewati preprocessing, yaitu setelah baris dengan data wajib kosong dibuang dan baris duplicate dihapus. Selisih 25 baris itu bukan data yang hilang secara tidak sengaja, tapi baris yang memang tidak memenuhi syarat minimal sebagai transaksi yang valid.

**2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep
Veracity.**


Dalam konteks Veracity, 490 baris lebih bisa dipercaya untuk dijadikan dasar analisis dibanding 515 baris. Veracity bukan soal jumlah data yang paling banyak, tapi seberapa akurat dan konsisten data itu mencerminkan kejadian yang sebenarnya. Dari 515 baris awal, sebagian adalah catatan ganda dari transaksi yang sama, sehingga kalau dihitung apa adanya akan membuat jumlah transaksi maupun total penjualan tercatat lebih besar dari kenyataan. Baris lain yang kehilangan customer_name atau payment_method juga tidak bisa dipakai untuk identifikasi transaksi secara utuh. Jadi 490 baris bukan berarti lebih sedikit datanya, tapi lebih murni mewakili transaksi yang benar-benar valid.

**3.Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing
value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?**


Rating memang tidak wajib diisi pembeli, sehingga cukup banyak baris yang kosong pada kolom ini. Kalau nilai kosong itu dipaksa diisi dengan angka tebakan, rating rata-rata yang dilaporkan justru akan menyesatkan karena bukan mencerminkan penilaian pembeli yang sesungguhnya. Cara paling jujur menjelaskannya ke tim Finance adalah dengan menyampaikan bahwa rata-rata rating dihitung hanya dari transaksi yang benar-benar memiliki rating, dan mencantumkan berapa jumlah transaksi yang dipakai sebagai dasar perhitungan tersebut, bukan dari seluruh 490 transaksi.

# **Q. Latihan**

## 1.  Ubah SEED menjadi 7 dan jalankan ulang seluruh pipeline. Bandingkan jumlah baris `transaksi_mentah.csv` dan `transaksi_bersih.csv` dengan hasil SEED = 42.

In [13]:
def jalankan_pipeline(seed):
    np.random.seed(seed)
    random.seed(seed)
    fk = Faker("id_ID")
    Faker.seed(seed)

    rows_ = []
    for i in range(1, N + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fk.name()
        produk = fk.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
        qty = random.randint(1, 5)
        harga_variants = [str(harga_dasar), f"Rp{harga_dasar:,}".replace(",", "."), f"{harga_dasar}.0", f" {harga_dasar} "]
        harga = random.choice(harga_variants)
        tgl = fk.date_between(start_date="-90d", end_date="today")
        tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
        tanggal = random.choice(tgl_variants)
        metode = random.choice(metode_bayar)
        if random.random() < 0.3:
            metode = metode.lower()
        if random.random() < 0.2:
            kategori = kategori.upper() + "  "
        kota = fk.city()
        rating = random.choice([1, 2, 3, 4, 5, None, None])
        rows_.append({"transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
                       "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
                       "transaction_date": tanggal, "shipping_city": kota, "rating": rating})

    d = pd.DataFrame(rows_)
    for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        idx = d.sample(frac=frac, random_state=seed).index
        d.loc[idx, col] = np.nan
    dup = d.sample(n=15, random_state=seed)
    d = pd.concat([d, dup], ignore_index=True)
    d = d.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_mentah = len(d)

    d = d.dropna(subset=["customer_name", "payment_method"])
    d["shipping_city"] = d["shipping_city"].fillna("Tidak Diketahui")
    d = d.drop_duplicates()
    n_bersih = len(d)
    return n_mentah, n_bersih

mentah_42, bersih_42 = 515, len(df)
mentah_7, bersih_7 = jalankan_pipeline(7)

print(f"SEED=42 -> transaksi_mentah: {mentah_42} baris, transaksi_bersih: {bersih_42} baris")
print(f"SEED=7  -> transaksi_mentah: {mentah_7} baris, transaksi_bersih: {bersih_7} baris")

SEED=42 -> transaksi_mentah: 515 baris, transaksi_bersih: 490 baris
SEED=7  -> transaksi_mentah: 515 baris, transaksi_bersih: 490 baris


Pipeline K-2 sampai K-4 dibungkus jadi satu fungsi supaya bisa dipanggil ulang dengan seed berapa pun, tanpa merusak variabel df yang sudah diproses di atas. Jadi hasil SEED 42 dan SEED 7 bisa dibandingkan langsung.

Jawabannya: jumlahnya sama. Data mentah sama-sama 515 baris, setelah dropna() sama-sama 495, dan dataset bersihnya sama-sama 490.

Awalnya terlihat seperti bakal beda, tapi setelah dicek ternyata masuk akal. Jumlah baris di tiap tahap itu ditentukan oleh angka yang sudah ditulis tetap di kode, bukan oleh hasil acaknya. N = 500 itu tetap, n=15 untuk duplikasi juga tetap, dan frac=0.02 artinya selalu 2% dari 1000 baris alias 20 baris, mau seed-nya berapa pun. Seed cuma menentukan baris mana yang terpilih dan isinya apa, bukan berapa banyak.

Satu yang beda cuma kolom rating: SEED 42 ada 166 yang kosong, SEED 7 ada 120. Ini karena rating diisi lewat random.choice([1,2,3,4,5,None,None]) yang murni acak per baris, tidak pakai proporsi tetap seperti kolom lain. Tapi karena rating tidak dipakai untuk dropna(), bedanya tidak berpengaruh ke jumlah baris akhir.

##2. Tambahkan kolom `is_valid_price` bernilai True jika price > 0.

In [14]:
df["is_valid_price"] = df["price"] > 0
print(df["is_valid_price"].value_counts())
print("Jumlah harga tidak valid:", (~df["is_valid_price"]).sum())

is_valid_price
True    490
Name: count, dtype: int64
Jumlah harga tidak valid: 0


df["price"] > 0 langsung dijalankan ke seluruh kolom sekaligus, jadi tidak perlu looping. Hasilnya kolom baru berisi True/False. Tanda ~ di baris terakhir untuk membalik nilainya, jadi yang dihitung malah yang harganya tidak valid.

Hasilnya semua 490 baris True, tidak ada harga yang nol atau minus. Ini sekalian jadi bukti kalau fungsi bersihkan_harga() di K-5b sudah jalan benar, soalnya kalau ada nilai yang gagal dikonversi pasti jadi NaN dan bakal ketahuan di sini.



##3. Hitung jumlah transaksi per category menggunakan value_counts().

In [15]:
print(df["category"].value_counts())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


value_counts() menghitung berapa kali tiap nilai muncul di kolom, lalu diurutkan dari yang paling banyak.

Hasilnya Olahraga 97, Kesehatan 91, Elektronik 89, Buku 82, Fashion 66, Rumah Tangga 65. Totalnya 490, cocok dengan jumlah baris dataset bersih. Sebarannya lumayan rata karena kategori dipilih acak pakai random.choice() waktu membuat dataset.

Hasil ini bisa rapi begini justru karena standardisasi teks sudah dilakukan duluan di K-5a. Kalau dijalankan di data mentah, "Elektronik", "ELEKTRONIK ", dan "elektronik" akan terhitung jadi tiga kategori berbeda.